# 🏗️ Delta Lake — Marketplace de Máquinas Pesadas

## 📋 Cenário

Este notebook demonstra o uso do **Delta Lake** com **Apache Spark (PySpark)** em um cenário de **Marketplace de Máquinas Pesadas**.

Simulamos uma plataforma digital onde vendedores anunciam máquinas como escavadeiras, retroescavadeiras e pás carregadeiras para potenciais compradores.

## 🎯 O que será demonstrado

1. Configuração da `SparkSession` com suporte ao Delta Lake
2. Criação de tabela com `USING delta`
3. Operações DML: `INSERT`, `SELECT`, `UPDATE`, `DELETE`
4. Histórico de transações com `DESCRIBE HISTORY`

## 🗃️ Estrutura da Tabela `maquinas_delta`

| Campo | Tipo | Descrição |
|-------|------|----------|
| `id` | INT | Identificador único do anúncio |
| `tipo` | STRING | Tipo da máquina (Escavadeira, Retroescavadeira...) |
| `marca` | STRING | Fabricante (Caterpillar, JCB, Volvo...) |
| `modelo` | STRING | Modelo específico (320D, 3CX, L90F...) |
| `ano` | INT | Ano de fabricação |
| `preco` | DOUBLE | Preço de venda em R$ |
| `status` | STRING | Estado: `disponivel`, `em_negociacao`, `vendido` |

---

## ⚙️ 1. Configuração da SparkSession

Criamos uma `SparkSession` configurada com as extensões necessárias para o **Delta Lake**:

- **`io.delta:delta-spark_2.12:3.2.0`** — biblioteca Delta Lake para Spark 3.5 / Scala 2.12
- **`DeltaSparkSessionExtension`** — habilita sintaxe SQL do Delta (`DESCRIBE HISTORY`, `MERGE INTO`)
- **`DeltaCatalog`** — substitui o catálogo padrão do Spark pelo catálogo Delta

> ⏱️ **Nota:** Na **primeira execução**, o Spark fará o download dos JARs do Maven Central (~30–60s).
> As execuções seguintes serão mais rápidas pois os JARs ficam em cache em `~/.ivy2/`.

In [1]:
import os

# Configurações obrigatórias no Windows para o Spark funcionar
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot'
os.environ['SPARK_HOME'] = r'C:\spark-work\.venv\Lib\site-packages\pyspark'
os.environ['PYSPARK_PYTHON'] = r'C:\spark-work\.venv\Scripts\python.exe'
os.environ['HADOOP_HOME'] = r'C:\hadoop'

from pyspark.sql import SparkSession

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession
    .builder
    .appName("TrabalhoDeltaLake")
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.sql.warehouse.dir", r"C:\spark-work\spark-warehouse\delta")
    .config("spark.hadoop.io.nativeio.enabled", "false")
    .getOrCreate()
)

print(f"✅ SparkSession iniciada com sucesso!")
print(f"   Versão do Spark  : {spark.version}")
print(f"   App Name         : {spark.sparkContext.appName}")
print(f"   Master           : {spark.sparkContext.master}")
print(f"   Spark UI         : http://localhost:4040")

✅ SparkSession iniciada com sucesso!
   Versão do Spark  : 3.5.3
   App Name         : TrabalhoDeltaLake
   Master           : local[*]
   Spark UI         : http://localhost:4040


---

## 🗄️ 2. Criação da Tabela Delta

Primeiro removemos a tabela se ela já existir (para garantir execução **idempotente** do notebook),
depois criamos a tabela com `USING delta`.

O Delta Lake vai criar automaticamente:
- O diretório `spark-warehouse/delta/maquinas_delta/`
- O subdiretório `_delta_log/` com o arquivo `00000000000000000000.json` (versão 0: CREATE TABLE)

In [2]:
spark.sql("DROP TABLE IF EXISTS maquinas_delta")
print("🗑️  Tabela anterior removida (se existia)")

spark.sql("""
    CREATE TABLE maquinas_delta (
        id     INT,
        tipo   STRING,
        marca  STRING,
        modelo STRING,
        ano    INT,
        preco  DOUBLE,
        status STRING
    )
    USING delta
""")

print("✅ Tabela 'maquinas_delta' criada com formato Delta Lake!")

🗑️  Tabela anterior removida (se existia)


✅ Tabela 'maquinas_delta' criada com formato Delta Lake!


---

## ➕ 3. Inserção de Dados (INSERT)

Inserimos 3 anúncios de máquinas pesadas no marketplace.

Cada `INSERT` é uma **transação ACID** — ou todos os registros são inseridos com sucesso,
ou nenhum é (em caso de falha). O Delta Lake registra esta operação como a **versão 1** no `_delta_log`.

In [3]:
spark.sql("""
    INSERT INTO maquinas_delta VALUES
        (1, 'Escavadeira',      'Caterpillar', '320D', 2018, 350000.00, 'disponivel'),
        (2, 'Retroescavadeira', 'JCB',         '3CX',  2020, 280000.00, 'disponivel'),
        (3, 'Pa Carregadeira',  'Volvo',       'L90F', 2017, 420000.00, 'disponivel')
""")

print("✅ 3 máquinas inseridas com sucesso!")

✅ 3 máquinas inseridas com sucesso!


---

## 🔍 4. Consulta Inicial (SELECT)

Verificamos os dados inseridos. Todos os 3 anúncios estão com status `disponivel`.

In [4]:
print("📋 Estado inicial da tabela maquinas_delta:")
spark.sql("SELECT * FROM maquinas_delta ORDER BY id").show(truncate=False)

📋 Estado inicial da tabela maquinas_delta:


+---+----------------+-----------+------+----+--------+----------+
|id |tipo            |marca      |modelo|ano |preco   |status    |
+---+----------------+-----------+------+----+--------+----------+
|1  |Escavadeira     |Caterpillar|320D  |2018|350000.0|disponivel|
|2  |Retroescavadeira|JCB        |3CX   |2020|280000.0|disponivel|
|3  |Pa Carregadeira |Volvo      |L90F  |2017|420000.0|disponivel|
+---+----------------+-----------+------+----+--------+----------+



---

## ✏️ 5. Atualização de Dados (UPDATE)

A **JCB 3CX** (id=2) entrou em processo de negociação. O comprador fez uma proposta de R$ 265.000,00.
Atualizamos o preço e o status do anúncio.

**Como o Delta Lake executa o UPDATE internamente:**

1. Lê o arquivo Parquet que contém o registro `id = 2`
2. Gera um **novo arquivo Parquet** com o registro atualizado
3. Marca o arquivo antigo como `remove` no `_delta_log`
4. Registra a operação `UPDATE` como a **versão 2**

In [5]:
spark.sql("""
    UPDATE maquinas_delta
    SET preco  = 265000.00,
        status = 'em_negociacao'
    WHERE id = 2
""")

print("✅ Registro id=2 atualizado — JCB 3CX em negociação por R$ 265.000,00")

✅ Registro id=2 atualizado — JCB 3CX em negociação por R$ 265.000,00


In [6]:
print("📋 Estado da tabela após UPDATE:")
spark.sql("SELECT * FROM maquinas_delta ORDER BY id").show(truncate=False)

📋 Estado da tabela após UPDATE:


+---+----------------+-----------+------+----+--------+-------------+
|id |tipo            |marca      |modelo|ano |preco   |status       |
+---+----------------+-----------+------+----+--------+-------------+
|1  |Escavadeira     |Caterpillar|320D  |2018|350000.0|disponivel   |
|2  |Retroescavadeira|JCB        |3CX   |2020|265000.0|em_negociacao|
|3  |Pa Carregadeira |Volvo      |L90F  |2017|420000.0|disponivel   |
+---+----------------+-----------+------+----+--------+-------------+



---

## 🗑️ 6. Exclusão de Dados (DELETE)

A **Volvo L90F** (id=3) foi vendida. O anúncio será removido da plataforma.

Assim como o UPDATE, o DELETE não apaga fisicamente o arquivo Parquet — ele apenas registra
o arquivo como `remove` no `_delta_log` (versão 3). O arquivo físico permanece para
possibilitar o **Time Travel**.

In [7]:
spark.sql("""
    DELETE FROM maquinas_delta
    WHERE id = 3
""")

print("✅ Registro id=3 removido — Volvo L90F foi vendida")

✅ Registro id=3 removido — Volvo L90F foi vendida


In [8]:
print("📋 Estado final da tabela após DELETE:")
spark.sql("SELECT * FROM maquinas_delta ORDER BY id").show(truncate=False)

📋 Estado final da tabela após DELETE:


+---+----------------+-----------+------+----+--------+-------------+
|id |tipo            |marca      |modelo|ano |preco   |status       |
+---+----------------+-----------+------+----+--------+-------------+
|1  |Escavadeira     |Caterpillar|320D  |2018|350000.0|disponivel   |
|2  |Retroescavadeira|JCB        |3CX   |2020|265000.0|em_negociacao|
+---+----------------+-----------+------+----+--------+-------------+



---

## 📜 7. Histórico de Transações (DESCRIBE HISTORY)

O Delta Lake registra **todas as operações** realizadas na tabela no `_delta_log`.
O comando `DESCRIBE HISTORY` exibe esse histórico completo.

**Colunas mais importantes:**

| Coluna | Descrição |
|--------|----------|
| `version` | Número sequencial da versão (começa em 0) |
| `timestamp` | Quando a operação foi confirmada |
| `operation` | Tipo da operação (CREATE TABLE, WRITE, UPDATE, DELETE) |
| `operationParameters` | Parâmetros como predicados de filtro e modo de escrita |
| `numOutputRows` | Número de linhas afetadas |

> 💡 Com o histórico, você pode usar **Time Travel** para consultar qualquer versão:
> ```python
> spark.read.format("delta").option("versionAsOf", 1).table("maquinas_delta").show()
> ```

In [9]:
print("📜 Histórico completo de transações da tabela maquinas_delta:")
spark.sql("DESCRIBE HISTORY maquinas_delta").show(truncate=False)

📜 Histórico completo de transações da tabela maquinas_delta:


+-------+-----------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation   |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                                                          

---

## ✅ 8. Conclusão

Demonstramos com sucesso as principais funcionalidades do **Delta Lake** no cenário do Marketplace de Máquinas:

| Operação | Resultado |
|----------|----------|
| SparkSession com Delta Lake | ✅ Configurada com extensões e catálogo Delta |
| `CREATE TABLE ... USING delta` | ✅ Tabela criada com `_delta_log` |
| `INSERT INTO` | ✅ 3 anúncios inseridos (versão 1) |
| `SELECT` | ✅ Dados consultados |
| `UPDATE` | ✅ JCB 3CX atualizada (versão 2) |
| `DELETE` | ✅ Volvo L90F removida (versão 3) |
| `DESCRIBE HISTORY` | ✅ Histórico de 4 versões exibido |

### 🗂️ Arquivos gerados em `spark-warehouse/delta/maquinas_delta/`

```
maquinas_delta/
├── _delta_log/
│   ├── 00000000000000000000.json  ← CREATE TABLE
│   ├── 00000000000000000001.json  ← INSERT
│   ├── 00000000000000000002.json  ← UPDATE
│   └── 00000000000000000003.json  ← DELETE
└── *.snappy.parquet               ← Arquivos de dados
```

In [10]:
spark.stop()
print("🛑 SparkSession encerrada com sucesso.")

🛑 SparkSession encerrada com sucesso.
